In [1]:
# Mô tả: Cấu hình biến môi trường và số luồng cho BLAS/TF
import os

os.environ['OPENBLAS_NUM_THREADS'] = '44'  # 50% cores
os.environ['MKL_NUM_THREADS'] = '44'
os.environ['OMP_NUM_THREADS'] = '44'
os.environ['NUMEXPR_NUM_THREADS'] = '44'

# TensorFlow threading
os.environ['TF_NUM_INTRAOP_THREADS'] = '44'  # Parallel ops
os.environ['TF_NUM_INTEROP_THREADS'] = '8'   # Independent ops

# turn off oneDNN optimization if needed
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

print("Configured for 88-core CPU")

Configured for 88-core CPU


In [2]:
from pathlib import Path

# [va lai 2026-09-17] 03 da chuyen tu Code/ vao Code/Code_SDC_V1/. Chi doi cach DO
# duong dan, khong dong vao bat ky logic model nao.
_cwd = Path.cwd().resolve()
_NB03 = next((hit for p in (_cwd, *_cwd.parents)
              for hit in sorted((p / "Code").glob("**/03_train_model.ipynb"))), None)
assert _NB03 is not None, f"Không thấy 03_train_model.ipynb dưới Code/ quanh {_cwd}"

if not globals().get("SDC_DEFS_LOADED"):
    SDC_IMPORT_ONLY = True
    try:
        get_ipython().run_line_magic("run", f'-i "{_NB03}"')
    finally:
        del SDC_IMPORT_ONLY

import json
from datetime import datetime
from collections import Counter, OrderedDict

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

Configured for 88-core CPU
Gốc dự án: C:\Users\admin\Desktop\02.SDC
Đã nạp hàm SDC từ 03_train_model.ipynb


## Tập train: 12 thiết bị FIELD làm lớp thật, phần còn lại làm nền `__unknown__`

Danh sách thiết bị FIELD lấy động từ cột `scenario` — không hardcode tên, để notebook
này vẫn đúng khi bạn capture thêm thiết bị/ngày FIELD mới.

In [3]:
UNKNOWN_LABEL = "__unknown__"

sessions, feature_groups = load_sessions(SESSIONS_PATH)
features = feature_groups["all"]

is_field = sessions["scenario"].eq("FIELD")
FIELD_DEVICES = sorted(sessions.loc[is_field, "canonical_device"].unique())

print(f"Tổng {len(sessions)} dòng — {is_field.sum()} dòng / {len(FIELD_DEVICES)} thiết bị "
      f"FIELD làm lớp thật, {(~is_field).sum()} dòng CIC-2022 làm nền {UNKNOWN_LABEL!r}")
for head in LABEL_COLS:
    vc = sessions.loc[is_field, head].value_counts()
    print(f"  {head}: {len(vc)} lớp — {dict(vc)}")

Tổng 8189 dòng — 50 dòng / 12 thiết bị FIELD làm lớp thật, 8139 dòng CIC-2022 làm nền '__unknown__'
  make: 8 lớp — {'Generic Laptop': np.int64(22), 'Camera': np.int64(6), 'Linova/Linux': np.int64(6), 'Samsung': np.int64(6), 'Raspberry Pi': np.int64(3), 'Xiaomi': np.int64(3), 'Apple': np.int64(3), 'OPPO': np.int64(1)}
  type: 4 lớp — {'Laptop': np.int64(28), 'Smartphone': np.int64(13), 'IP Camera': np.int64(6), 'Single-board Computer': np.int64(3)}
  model: 11 lớp — {'Windows Desktop DELL': np.int64(11), 'Generic IP Camera': np.int64(6), 'Windows Laptop HP': np.int64(6), 'Linova Laptop HP': np.int64(6), 'Samsung Galaxy': np.int64(6), 'Windows Desktop HP': np.int64(5), 'Raspberry Pi': np.int64(3), 'iPhone': np.int64(3), 'Redmi Note 14 Pro': np.int64(2), 'OPPO A92': np.int64(1), 'Redmi Note 10': np.int64(1)}


## Fit encoder + model (250 cây, giống cấu hình đang dùng ở `03`)

Encoder vẫn fit trên **toàn bộ** dữ liệu (kể cả CIC-2022) để vocab TF-IDF phủ được cả
nền — fit riêng trên 50 dòng FIELD thì DNS/TLS token của CIC-2022 thành out-of-vocab hết,
"trông khác lạ" mất nghĩa.

Nhưng fit trên toàn bộ **kèm trần `max_features`** thì hỏng theo chiều ngược lại, và đó
là lý do model cũ không nhận được thiết bị lúc chạy thật: TF-IDF cắt vocab theo tần suất
toàn corpus, mà 8139/8189 dòng là CIC-2022, nên token nhận dạng của 12 thiết bị FIELD
thua cuộc đua tần suất và bị đá ra. Đo trên bản `20260915_170620_field_closedset`:

| thiết bị | `dns_tokens` OOV | `tls_sni_tokens` OOV | token bị mất |
|---|---|---|---|
| Raspberry Pi | 74% | 75% | `mozilla` `firefox` `detectportal` `safebrowsing` `merino` `push` `services` |
| Linova Laptop | 71% | 48% | `archlinux` `vntek` `vnpt` `sharepoint` `chatgpt`(dns) |

Những cái bị mất đúng là loại tín hiệu ổn định nhất: `detectportal.firefox.com` và
`ping.archlinux.org` là connectivity-check tự động của Firefox / NetworkManager, phát ra
mỗi lần lên mạng chứ không phụ thuộc người dùng bấm gì. Còn lại trong vocab chỉ là
`com` / `org` / `googleapis` — không đủ để phân biệt gì, nên forest chỉ còn cách nhớ
thuộc lòng đúng 3–6 dòng train.

Xem cell dưới cho hai fix và phần chúng **không** chữa được.

In [4]:
# --- FIX 1: bỏ trần `max_features` của TF-IDF -------------------------------------
# `_make_vectorizer` ở 03 đọc `TFIDF_MAX_FEATURES` tại thời điểm fit, nên ghi đè ở đây là
# đủ — không đụng vào contract `sdc-tiered-v2` mà 03/05/06/07 đang dùng chung.
#
# Trần 200/100/150/100 là hợp lý khi 12 lớp đều có hàng trăm phiên như ở 03. Ở đây tập
# FIELD chỉ 50 dòng, nên trần đó cắt đúng vào phần cần giữ (xem markdown trên). Bỏ trần
# chỉ nở input tensor 518 -> 1088 feature, vẫn thừa sức chạy trên router.
TFIDF_MAX_FEATURES = {col: None for col in TFIDF_MAX_FEATURES}

encoder = fit_encoder(sessions, features)
X_all, feature_names, _ = apply_encoder(sessions, encoder)
print(f"vocab sau khi bỏ trần: {X_all.shape[1]} feature — "
      + ", ".join(f"{c}={len(encoder['ct'].named_transformers_['t_' + c].vocabulary_)}"
                  for c in encoder["text_cols"]))

# --- FIX 2: không ép dòng non-FIELD mang nhãn thật thành `__unknown__` -------------
# `np.where(is_field, ..., UNKNOWN_LABEL)` gán TẤT CẢ dòng CIC-2022 thành `__unknown__`,
# kể cả những dòng mang đúng cái nhãn mà một thiết bị FIELD đang giữ. Head `type` dính
# nặng: 2471 dòng CIC-2022 có `type == "IP Camera"` thật, chọi 6 dòng FIELD cùng nhãn —
# forest được dạy hai điều ngược nhau trên cùng một nhãn, tỉ lệ 412:1.
#
# Bỏ hẳn các dòng đó khỏi tập train của head đang xét, thay vì dán nhãn sai cho chúng.
# Nền `__unknown__` của head `type` vẫn còn 5668 dòng nên khả năng từ chối không mất:
# đo lại trên CIC-2022 vẫn 0/8139 báo nhầm, IoT Sentinel vẫn 510/510 từ chối đúng.
models = {}
train_rows = {}
for head in LABEL_COLS:
    y = np.where(is_field, sessions[head].astype(str), UNKNOWN_LABEL)
    field_labels = set(sessions.loc[is_field, head].astype(str))
    clash = (~is_field) & sessions[head].astype(str).isin(field_labels)
    keep = ~clash.to_numpy()
    train_rows[head] = int(keep.sum())

    m = make_model()
    m.fit(X_all[keep], y[keep])
    models[head] = m
    print(f"  fit {head}: {len(m.classes_)} lop (gom {UNKNOWN_LABEL!r}), "
          f"{keep.sum()} dong train, bo {(~keep).sum()} dong non-FIELD trung nhan that")

# --- Hai fix này KHÔNG chữa được cái gì --------------------------------------------
# Không chữa được việc lớp chỉ có 1-6 phiên, tất cả từ MỘT ngày capture và MỘT MAC.
# `RARE_THRESHOLD = 10` ở 03 nói thẳng "lớp dưới ngưỡng này không học được"; notebook này
# đang train Raspberry Pi (3 phiên), iPhone (3), Redmi Note 14 Pro (2), OPPO A92 (1),
# Redmi Note 10 (1) bất chấp ngưỡng đó. Với `class_weight="balanced"` mỗi phiên Pi nặng
# gấp ~2700 lần một phiên `__unknown__`, nên forest khoét đúng 3 ô hẹp quanh 3 điểm đó.
#
# Riêng Linova Laptop còn một vấn đề dữ liệu mà không fix code nào chạm tới được: nó
# KHÔNG có feature nội tại nào khác 4 laptop FIELD còn lại — cùng `tls_fp`
# (0303|4865,4866,4867,...|h2,http/1.1, dùng chung với Desktop PC DNGJHRT / DucAnh /
# VAF70SQ6 / Lee Kingdom), không mDNS, không `dhcp_vci`, `dhcp_prl` trùng hệt Raspberry
# Pi. Thứ duy nhất tách nó ra là tập token DNS/SNI của người dùng. Đã thử bỏ trần +
# binary/norm=None: Linova vẫn `__unknown__` ngay khi đổi tập duyệt web.
print(f"\nSố phiên mỗi lớp (nhắc lại — RARE_THRESHOLD={RARE_THRESHOLD}):")
for head in LABEL_COLS:
    thin = sessions.loc[is_field, head].value_counts()
    thin = thin[thin < RARE_THRESHOLD]
    print(f"  {head}: {dict(thin)}")

vocab sau khi bỏ trần: 1088 feature — dhcp_vci=32, dns_tokens=575, mdns_tokens=118, tls_sni_tokens=327
  fit make: 9 lop (gom '__unknown__'), 8189 dong train, bo 0 dong non-FIELD trung nhan that
  fit type: 5 lop (gom '__unknown__'), 5718 dong train, bo 2471 dong non-FIELD trung nhan that
  fit model: 12 lop (gom '__unknown__'), 8189 dong train, bo 0 dong non-FIELD trung nhan that

Số phiên mỗi lớp (nhắc lại — RARE_THRESHOLD=10):
  make: {'Camera': np.int64(6), 'Linova/Linux': np.int64(6), 'Samsung': np.int64(6), 'Raspberry Pi': np.int64(3), 'Xiaomi': np.int64(3), 'Apple': np.int64(3), 'OPPO': np.int64(1)}
  type: {'IP Camera': np.int64(6), 'Single-board Computer': np.int64(3)}
  model: {'Generic IP Camera': np.int64(6), 'Windows Laptop HP': np.int64(6), 'Linova Laptop HP': np.int64(6), 'Samsung Galaxy': np.int64(6), 'Windows Desktop HP': np.int64(5), 'Raspberry Pi': np.int64(3), 'iPhone': np.int64(3), 'Redmi Note 14 Pro': np.int64(2), 'OPPO A92': np.int64(1), 'Redmi Note 10': np.int64

## Đóng gói + lưu

Định dạng riêng `sdc-closedset-v1` — **không tương thích** với `Predictor` (contract
`sdc-tiered-v2`) ở `03`/`06`: không có bảng L1, không có ngưỡng theo `n_sources`, vì kiến
trúc này không cần — bản thân lớp `__unknown__` đã là cơ chế từ chối.

In [5]:
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_field_closedset")
run_dir = MODELS / RUN_ID
run_dir.mkdir(parents=True)

bundle = {
    "format": "sdc-closedset-v1",
    "unknown_label": UNKNOWN_LABEL,
    "heads": list(LABEL_COLS),
    "feature_cols": features,
    "feature_names": feature_names,
    "encoder": encoder,
    "models": models,
    "field_devices": FIELD_DEVICES,
}
joblib.dump(bundle, run_dir / "model.joblib", compress=3)
(run_dir / "meta.json").write_text(json.dumps({
    "format": "sdc-closedset-v1",
    "run_id": RUN_ID,
    "created": datetime.now().isoformat(timespec="seconds"),
    "dataset": str(SESSIONS_PATH),
    "n_sessions": int(len(sessions)),
    "n_field_devices": len(FIELD_DEVICES),
    "field_devices": FIELD_DEVICES,
    # Đọc thẳng từ model đã fit, không dựng lại y để đếm: sau FIX 2 mỗi head có tập train
    # riêng, đếm lại bằng công thức cũ là ghi vào meta một con số không phải của model này.
    "heads": {h: int(len(models[h].classes_)) for h in LABEL_COLS},
    "train_rows": train_rows,
    "n_features": int(X_all.shape[1]),
    "tfidf_max_features": TFIDF_MAX_FEATURES,
    "note": ("Closed-set: 12 thiết bị FIELD là lớp thật, CIC-2022 làm nền __unknown__. "
             "FIX 1: bỏ trần TF-IDF max_features (trần cũ cắt mất 71-75% token nhận dạng "
             "của FIELD vì vocab bị 8139 dòng CIC-2022 chiếm chỗ). FIX 2: dòng non-FIELD "
             "mang đúng nhãn của một lớp FIELD bị loại khỏi tập train của head đó thay vì "
             "ép thành __unknown__ (head type: 2471 dòng IP Camera). "
             "CẢNH BÁO CHƯA GỠ: đánh giá FIELD ở notebook này là in-sample, 1 ngày capture, "
             "1 MAC mỗi thiết bị. Các lớp dưới RARE_THRESHOLD=10 (Raspberry Pi 3 phiên, "
             "iPhone 3, Redmi Note 14 Pro 2, OPPO A92 1, Redmi Note 10 1) là ghi nhớ thuộc "
             "lòng, không phải khái quát. Linova Laptop không có feature nội tại nào khác "
             "4 laptop FIELD còn lại (chung tls_fp, không mDNS, không dhcp_vci) — chỉ tách "
             "được bằng token DNS/SNI của người dùng, nên sẽ trượt lúc chạy thật."),
}, indent=2, ensure_ascii=False), encoding="utf-8")

print("Đã lưu:", run_dir)

Đã lưu: C:\Users\admin\Desktop\02.SDC\Models\20260917_105135_field_closedset


## Predictor tối giản cho closed-set

Không cần bảng L1 hay ngưỡng theo nguồn bằng chứng — chỉ argmax `predict_proba`.
`is_unknown=True` khi lớp thắng là `__unknown__`.

In [6]:
class ClosedSetPredictor:
    def __init__(self, run_dir):
        bundle = joblib.load(Path(run_dir) / "model.joblib")
        assert bundle["format"] == "sdc-closedset-v1"
        self.unknown_label = bundle["unknown_label"]
        self.heads = bundle["heads"]
        self.feature_cols = bundle["feature_cols"]
        self.encoder = bundle["encoder"]
        self.models = bundle["models"]
        self.field_devices = bundle["field_devices"]

    def predict_row(self, row):
        frame = pd.DataFrame([row])[self.feature_cols]
        X, _, _ = apply_encoder(frame, self.encoder)
        out = {}
        for head in self.heads:
            proba = self.models[head].predict_proba(X)[0]
            classes = self.models[head].classes_
            idx = proba.argmax()
            top1, conf = classes[idx], float(proba[idx])
            out[head] = {"top1": top1, "confidence": conf,
                         "is_unknown": top1 == self.unknown_label}
        return out

    def predict_device(self, rows):
        """Gộp nhiều phiên/cửa sổ của cùng một thiết bị bằng bỏ phiếu đa số trên top1
        (đơn giản hơn DeviceTracker ở 03 — không cần ratio/ambiguous vì __unknown__ đã
        gánh vai trò từ chối)."""
        votes = {h: Counter() for h in self.heads}
        for row in rows:
            out = self.predict_row(row)
            for head in self.heads:
                votes[head][out[head]["top1"]] += 1
        result = {}
        for head in self.heads:
            top1, n = votes[head].most_common(1)[0]
            result[head] = {"top1": top1, "n_votes": n, "n_total": len(rows),
                             "is_unknown": top1 == self.unknown_label}
        return result


predictor = ClosedSetPredictor(run_dir)
print("Nạp lại từ", run_dir, "— OK")

Nạp lại từ C:\Users\admin\Desktop\02.SDC\Models\20260917_105135_field_closedset — OK


## Đánh giá

1. **Tự soi FIELD** — in-sample, chỉ để kiểm tra model học được (không phải bằng chứng
   khái quát, xem cảnh báo ở đầu notebook).
2. **IoT Sentinel** — 29 thiết bị hoàn toàn chưa đưa vào lúc fit, đây mới là phép thử
   trung thực cho khả năng từ chối "ngoài list".
3. **Biên độ xác suất** — kiểm tra thắng có sát nút không.

In [7]:
def eval_rows(sub_frame, label, is_field_data):
    Xs, _, _ = apply_encoder(sub_frame, predictor.encoder)
    print(f"\n=== {label} ({len(sub_frame)} dòng) ===")
    for head in predictor.heads:
        proba = predictor.models[head].predict_proba(Xs)
        classes = predictor.models[head].classes_
        top_idx = proba.argmax(axis=1)
        top1 = classes[top_idx]
        if is_field_data:
            truth = sub_frame[head].astype(str).to_numpy()
            correct = (top1 == truth).sum()
            to_unknown = (top1 == predictor.unknown_label).sum() - (truth == predictor.unknown_label).sum()
            other_wrong = len(sub_frame) - correct - to_unknown
            print(f"  {head:6s} đúng: {correct:4d}/{len(sub_frame)}   "
                  f"sai->__unknown__: {to_unknown:4d}   sai->lớp khác: {other_wrong:4d}")
        else:
            rejected = (top1 == predictor.unknown_label).sum()
            leaked = len(sub_frame) - rejected
            print(f"  {head:6s} từ chối đúng (__unknown__): {rejected:4d}/{len(sub_frame)}   "
                  f"BÁO NHẦM (gán vào 1 trong {len(FIELD_DEVICES)} thiết bị): {leaked:4d}")


eval_rows(sessions[is_field], "FIELD tự soi (in-sample — xem cảnh báo ở đầu notebook)",
          is_field_data=True)


=== FIELD tự soi (in-sample — xem cảnh báo ở đầu notebook) (50 dòng) ===
  make   đúng:   50/50   sai->__unknown__:    0   sai->lớp khác:    0
  type   đúng:   50/50   sai->__unknown__:    0   sai->lớp khác:    0
  model  đúng:   50/50   sai->__unknown__:    0   sai->lớp khác:    0


In [8]:
def load_windows(path):
    by_device = OrderedDict()
    with path.open(encoding="utf-8") as fh:
        for line in fh:
            w = json.loads(line)
            by_device.setdefault(w["device"], []).append(w)
    return by_device


SENTINEL_OUT = ROOT / "test_model" / "data_test" / "out"
by_device = load_windows(SENTINEL_OUT / "windows.jsonl")

rows = []
for device, windows in by_device.items():
    for w in windows:
        row = aggregate(w["records"], features)
        rows.append(row)
sentinel_frame = pd.DataFrame(rows)

eval_rows(sentinel_frame,
          f"IoT Sentinel ({len(by_device)} thiết bị lạ, chưa từng đưa vào lúc fit)",
          is_field_data=False)


=== IoT Sentinel (29 thiết bị lạ, chưa từng đưa vào lúc fit) (510 dòng) ===
  make   từ chối đúng (__unknown__):  510/510   BÁO NHẦM (gán vào 1 trong 12 thiết bị):    0
  type   từ chối đúng (__unknown__):  510/510   BÁO NHẦM (gán vào 1 trong 12 thiết bị):    0
  model  từ chối đúng (__unknown__):  510/510   BÁO NHẦM (gán vào 1 trong 12 thiết bị):    0


In [9]:
def margin_report(sub_frame, label):
    Xs, _, _ = apply_encoder(sub_frame, predictor.encoder)
    print(f"\n=== Biên độ xác suất — {label} ===")
    for head in predictor.heads:
        proba = predictor.models[head].predict_proba(Xs)
        classes = predictor.models[head].classes_
        unk_idx = list(classes).index(predictor.unknown_label)
        p_unknown = proba[:, unk_idx]
        other_idx = [i for i in range(len(classes)) if i != unk_idx]
        p_best_other = proba[:, other_idx].max(axis=1)
        print(f"  {head:6s} p(__unknown__): min={p_unknown.min():.3f} mean={p_unknown.mean():.3f}   "
              f"p(lớp thật cao nhất): max={p_best_other.max():.3f} mean={p_best_other.mean():.3f}")


margin_report(sentinel_frame, "IoT Sentinel (ngoài list)")
margin_report(sessions[is_field], "FIELD (trong list, in-sample)")


=== Biên độ xác suất — IoT Sentinel (ngoài list) ===
  make   p(__unknown__): min=0.684 mean=0.921   p(lớp thật cao nhất): max=0.120 mean=0.039
  type   p(__unknown__): min=0.728 mean=0.891   p(lớp thật cao nhất): max=0.164 mean=0.079
  model  p(__unknown__): min=0.688 mean=0.919   p(lớp thật cao nhất): max=0.132 mean=0.043

=== Biên độ xác suất — FIELD (trong list, in-sample) ===
  make   p(__unknown__): min=0.000 mean=0.000   p(lớp thật cao nhất): max=1.000 mean=1.000
  type   p(__unknown__): min=0.000 mean=0.000   p(lớp thật cao nhất): max=1.000 mean=1.000
  model  p(__unknown__): min=0.000 mean=0.000   p(lớp thật cao nhất): max=1.000 mean=1.000


## Xuất ONNX — một input / một output

In [10]:
import warnings

import onnx
import onnxruntime as ort
from onnx import TensorProto as TP
from onnx import compose, helper
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType, StringTensorType

ONNX_INPUT = "input"
ONNX_OUTPUT = "output"
ONNX_FILE = "sdc_closedset.onnx"
ONNX_CONTRACT_VERSION = "1.0.0"

ONNX_OPSET = 20
ONNX_ML_OPSET = 3

# Xem giải thích đầy đủ ở 07_export_onnx.ipynb (mục "Lắp ráp"): thiếu locale thì
# onnxruntime dựng `en_US.UTF-8` lúc nạp model và chết trên musl (router OpenWrt).
ONNX_LOCALE = "C"

### Bộ dựng graph + front

Rút gọn từ `OnnxGraphBuilder` ở 07 — chỉ giữ `name`/`add`/`const`, bỏ `label_encode`/
`concat_str` vì không cần dựng khoá L1 ở đây.

In [11]:
class OnnxGraphBuilder:
    """Gom node + initializer cho một graph, tự sinh tên không đụng nhau."""

    def __init__(self, prefix="sdc"):
        self.prefix = prefix
        self.nodes = []
        self.inits = []
        self._n = 0

    def name(self, stem):
        self._n += 1
        return f"{self.prefix}/{stem}_{self._n}"

    def add(self, op_type, inputs, out_stem, **attrs):
        out = self.name(out_stem)
        domain = attrs.pop("domain", "")
        self.nodes.append(helper.make_node(op_type, list(inputs), [out],
                                           name=self.name(f"n_{op_type}"),
                                           domain=domain, **attrs))
        return out

    def const(self, array, stem, dtype=None):
        out = self.name(stem)
        if dtype == TP.STRING:
            values = np.asarray(array, dtype=object)
            self.inits.append(helper.make_tensor(
                out, TP.STRING, list(values.shape),
                [s.encode("utf-8") for s in values.ravel()]))
        else:
            self.inits.append(onnx.numpy_helper.from_array(np.asarray(array), out))
        return out


def onnx_build_front(gb, cols, num_cols):
    """`input` string [N,len(cols)] -> dict {tên cột: tensor}. Cột numeric Cast sang float32."""
    outs = [gb.name(f"col/{c}") for c in cols]
    gb.nodes.append(helper.make_node("Split", [ONNX_INPUT], outs,
                                     name=gb.name("n_Split"), axis=1,
                                     num_outputs=len(cols)))
    num = set(num_cols)
    return {col: (gb.add("Cast", [t], f"num/{col}", to=TP.FLOAT) if col in num else t)
            for col, t in zip(cols, outs)}


def onnx_relink(graph, rename):
    """Sao node + initializer của một subgraph, đổi tên input theo `rename`."""
    nodes = []
    for node in graph.node:
        copy = onnx.NodeProto()
        copy.CopyFrom(node)
        for i, name in enumerate(copy.input):
            if name in rename:
                copy.input[i] = rename[name]
        nodes.append(copy)
    return nodes, list(graph.initializer)

### Chuyển encoder + 3 rừng cây, ghi locale, cắt trọng số

`onnx_sub_models`, `onnx_prune`, `onnx_set_locale`/`onnx_assert_locale` giống hệt 07 —
không phụ thuộc contract tiered, chỉ cần `bundle["encoder"]`/`["models"]`/
`["feature_names"]`, những khoá bundle closed-set này cũng có.

In [12]:
def onnx_sub_models(bundle):
    enc = bundle["encoder"]
    initial = ([(c, FloatTensorType([None, 1])) for c in enc["num_cols"]]
               + [(c, StringTensorType([None, 1]))
                  for c in enc["cat_cols"] + enc["text_cols"]])
    ct = convert_sklearn(enc["ct"], "sdc_encoder", initial_types=initial,
                         target_opset=ONNX_OPSET)
    ct = compose.add_prefix(ct, "enc/", rename_inputs=False)

    n_feat = len(bundle["feature_names"])
    forests = {}
    for head in bundle["heads"]:
        clf = bundle["models"][head]
        m = convert_sklearn(clf, f"sdc_{head}",
                            initial_types=[("X", FloatTensorType([None, n_feat]))],
                            target_opset=ONNX_OPSET,
                            options={id(clf): {"zipmap": False}})
        forests[head] = compose.add_prefix(m, f"rf_{head}/")
    return ct, forests


def onnx_prune(model):
    """Bỏ trọng số lá bằng 0 và hai thuộc tính đang mang đúng giá trị mặc định."""
    stats = {"weights_kept": 0, "weights_dropped": 0, "attrs_dropped": []}
    for node in model.graph.node:
        if node.op_type not in ("TreeEnsembleClassifier", "TreeEnsembleRegressor"):
            continue
        att = {a.name: a for a in node.attribute}
        weights = np.asarray(att["class_weights"].floats, dtype=np.float32)
        keep = np.flatnonzero(weights != 0)
        stats["weights_kept"] += int(keep.size)
        stats["weights_dropped"] += int(weights.size - keep.size)
        for field in ("class_ids", "class_nodeids", "class_treeids"):
            values = [int(v) for v in np.asarray(att[field].ints)[keep]]
            del att[field].ints[:]
            att[field].ints.extend(values)
        values = [float(v) for v in weights[keep]]
        del att["class_weights"].floats[:]
        att["class_weights"].floats.extend(values)

        hit = att.get("nodes_hitrates")
        miss = att.get("nodes_missing_value_tracks_true")
        drop = ([("nodes_hitrates", hit)] if hit is not None
                and all(v == 1.0 for v in hit.floats) else [])
        drop += ([("nodes_missing_value_tracks_true", miss)] if miss is not None
                 and all(v == 0 for v in miss.ints) else [])
        for name, attr in drop:
            node.attribute.remove(attr)
            stats["attrs_dropped"].append(name)

    model.graph.ClearField("doc_string")
    for node in model.graph.node:
        node.ClearField("doc_string")
    stats["attrs_dropped"] = sorted(set(stats["attrs_dropped"]))
    return stats


def onnx_set_locale(model, locale=ONNX_LOCALE):
    patched = []
    for node in model.graph.node:
        if node.op_type != "StringNormalizer":
            continue
        for existing in [a for a in node.attribute if a.name == "locale"]:
            node.attribute.remove(existing)
        node.attribute.append(helper.make_attribute("locale", locale))
        patched.append(node.name)
    return patched


def onnx_assert_locale(model, locale=ONNX_LOCALE):
    for node in model.graph.node:
        if node.op_type != "StringNormalizer":
            continue
        got = {a.name: a for a in node.attribute}.get("locale")
        assert got is not None, f"{node.name}: thiếu thuộc tính locale"
        assert got.s.decode() == locale, (
            f"{node.name}: locale {got.s.decode()!r}, cần {locale!r}")

### Lắp graph

Không có `onnx_build_l1`/`onnx_build_head`/`onnx_build_hierarchy` như 07 — mỗi forest đã
tự trả lời nhãn cuối cùng (kể cả `__unknown__`), `Reshape` rồi `Concat` là xong.

In [13]:
def onnx_build_model(bundle, prune=True):
    """Dựng ModelProto hoàn chỉnh. Trả (model, thứ tự cột, thống kê tối ưu)."""
    enc = bundle["encoder"]
    cols = enc["num_cols"] + enc["cat_cols"] + enc["text_cols"]
    ct, forests = onnx_sub_models(bundle)

    gb = OnnxGraphBuilder()
    columns = onnx_build_front(gb, cols, enc["num_cols"])

    # rename_inputs=False cua compose.add_prefix van gan tien to trong ban skl2onnx nay,
    # nen noi theo VI TRI (cung thu tu voi initial_types = cols) thay vi theo ten.
    nodes, inits = onnx_relink(ct.graph, {i.name: columns[c] for i, c in zip(ct.graph.input, cols)})
    gb.nodes += nodes
    gb.inits += inits
    features = ct.graph.output[0].name

    final = []
    for head in bundle["heads"]:
        forest = forests[head]
        nodes, inits = onnx_relink(forest.graph, {forest.graph.input[0].name: features})
        gb.nodes += nodes
        gb.inits += inits
        raw_label = forest.graph.output[0].name    # zipmap=False -> (label, probabilities)
        label = gb.add("Reshape", [raw_label, gb.const(np.int64([-1, 1]), "shape")],
                       f"out/{head}")
        final.append(label)

    gb.nodes.append(helper.make_node("Concat", final, [ONNX_OUTPUT],
                                     name=gb.name("n_Concat"), axis=1))

    graph = helper.make_graph(
        gb.nodes, "sdc_closedset",
        [helper.make_tensor_value_info(ONNX_INPUT, TP.STRING, [None, len(cols)])],
        [helper.make_tensor_value_info(ONNX_OUTPUT, TP.STRING, [None, len(bundle["heads"])])],
        gb.inits)

    opsets = {"": ONNX_OPSET, "ai.onnx.ml": ONNX_ML_OPSET}
    for sub in (ct, *forests.values()):
        for imp in sub.opset_import:
            opsets[imp.domain] = max(opsets.get(imp.domain, 0), imp.version)
    model = helper.make_model(
        graph, opset_imports=[helper.make_opsetid(d, v) for d, v in opsets.items()])
    model.ir_version = 10
    model.doc_string = ""

    stats = onnx_prune(model) if prune else {}
    stats["locale_nodes"] = onnx_set_locale(model)
    onnx_assert_locale(model)
    onnx.checker.check_model(model)
    return model, cols, stats


def onnx_contract(bundle, cols, run_id, stats, payload):
    enc = bundle["encoder"]
    return {
        "format": "sdc-closedset-onnx-v1",
        "contract_version": ONNX_CONTRACT_VERSION,
        "run_id": run_id,
        "file": ONNX_FILE,
        "bytes": len(payload),
        "runtime": {"required": "onnxruntime",
                    "reason": "TF-IDF dùng com.microsoft.Tokenizer",
                    "opset": ONNX_OPSET, "ai.onnx.ml": ONNX_ML_OPSET,
                    "string_normalizer_locale": ONNX_LOCALE},
        "io": {
            "input": {"name": ONNX_INPUT, "type": "string", "shape": ["N", len(cols)],
                      "columns": cols,
                      "note": ("Cột numeric gửi dưới dạng chuỗi số ('1', '0'); nguồn "
                               "vắng mặt -> numeric '0', cat/text '<missing>'.")},
            "output": {"name": ONNX_OUTPUT, "type": "string",
                      "shape": ["N", len(bundle["heads"])],
                      "heads": list(bundle["heads"]),
                      "unknown_label": bundle["unknown_label"],
                      "note": ("__unknown__ là một lớp forest tự trả lời — không có tầng "
                               "policy nào ngoài graph quyết định việc này.")},
        },
        "labels": {h: [str(c) for c in bundle["models"][h].classes_] for h in bundle["heads"]},
        "field_devices": bundle["field_devices"],
        "columns": {"num": enc["num_cols"], "cat": enc["cat_cols"], "text": enc["text_cols"]},
        "size_optimization": stats,
    }


def onnx_export(run_dir, prune=True):
    run_dir = Path(run_dir)
    bundle = joblib.load(run_dir / "model.joblib")
    fmt = bundle["format"]
    assert fmt == "sdc-closedset-v1", f"format lạ: {fmt}"
    model, cols, stats = onnx_build_model(bundle, prune=prune)
    payload = model.SerializeToString()
    path = run_dir / ONNX_FILE
    path.write_bytes(payload)
    (run_dir / "contract.json").write_text(
        json.dumps(onnx_contract(bundle, cols, run_dir.name, stats, payload),
                   indent=2, ensure_ascii=False), encoding="utf-8")
    return path

### Xuất candidate

In [14]:
onnx_path = onnx_export(run_dir)
onnx_contract_doc = json.loads((run_dir / "contract.json").read_text(encoding="utf-8"))

_raw, _, _ = onnx_build_model(bundle, prune=False)
_raw_bytes = len(_raw.SerializeToString())
_stats = onnx_contract_doc["size_optimization"]

print(f"run             {run_dir.name}")
print(f"model.joblib    {(run_dir / 'model.joblib').stat().st_size:>9,} bytes")
print(f"onnx chưa cắt   {_raw_bytes:>9,} bytes")
print(f"onnx đã cắt     {onnx_path.stat().st_size:>9,} bytes  "
      f"(-{1 - onnx_path.stat().st_size / _raw_bytes:.1%})")
print(f"trọng số lá giữ {_stats['weights_kept']:,} / "
      f"{_stats['weights_kept'] + _stats['weights_dropped']:,}")
display(pd.DataFrame([onnx_contract_doc["io"]["input"], onnx_contract_doc["io"]["output"]],
                     index=["input", "output"])[["name", "type", "shape"]])

run             20260917_105135_field_closedset
model.joblib    1,632,595 bytes
onnx chưa cắt   4,229,665 bytes
onnx đã cắt     1,556,730 bytes  (-63.2%)
trọng số lá giữ 25,001 / 227,721


,name,type,shape
input,input,string,"[N, 40]"
output,output,string,"[N, 3]"


### Parity với `ClosedSetPredictor`

So trực tiếp nhãn ONNX với `ClosedSetPredictor.predict_row` trên **từng dòng** của toàn
bộ `sessions_verified.parquet` (cả FIELD lẫn nền CIC-2022) — lệch ở đây nghĩa là graph
không tái lập đúng model đã fit, không phải sai số cho phép.

In [15]:
def onnx_to_input(frame, cols):
    out = frame[cols].copy()
    for col in cols:
        values = out[col]
        out[col] = (values.astype(np.float32).astype(str)
                    if pd.api.types.is_numeric_dtype(values) else values.astype(str))
    return out.to_numpy(dtype=object)


def onnx_reference_closedset(frame, heads):
    rows = []
    for record in frame.to_dict("records"):
        out = predictor.predict_row(record)
        rows.append([out[h]["top1"] for h in heads])
    return np.array(rows, dtype=object)


def onnx_verify_closedset(run_dir, contract, limit=None, seed=0):
    cols = contract["io"]["input"]["columns"]
    heads = contract["io"]["output"]["heads"]
    frame = sessions
    if limit and limit < len(frame):
        frame = frame.sample(limit, random_state=seed).reset_index(drop=True)

    session = ort.InferenceSession(str(Path(run_dir) / contract["file"]),
                                   providers=["CPUExecutionProvider"])
    got = session.run(None, {contract["io"]["input"]["name"]:
                             onnx_to_input(frame, cols)})[0]
    want = onnx_reference_closedset(frame, heads)

    rows, mismatch = [], {}
    for i, head in enumerate(heads):
        same = got[:, i] == want[:, i]
        rows.append({"head": head, "n": len(frame), "khớp": float(same.mean()),
                     "lệch": int((~same).sum())})
        if not same.all():
            bad = frame.loc[~same, ["canonical_device", head]].copy()
            bad["onnx"] = got[~same, i]
            bad["predictor"] = want[~same, i]
            mismatch[head] = bad
    return pd.DataFrame(rows).set_index("head"), mismatch


onnx_parity, onnx_mismatch = onnx_verify_closedset(run_dir, onnx_contract_doc)
display(onnx_parity)
for head, bad in onnx_mismatch.items():
    print(f"\n{head}: {len(bad)} dòng lệch")
    display(bad.head(15))
assert not onnx_mismatch, "ONNX và ClosedSetPredictor không khớp — xem bảng lệch ở trên"
print("Parity OK — graph và ClosedSetPredictor cho cùng nhãn trên mọi dòng")

,n,khớp,lệch
head,,,
make,8189,1.0,0
type,8189,1.0,0
model,8189,1.0,0


Parity OK — graph và ClosedSetPredictor cho cùng nhãn trên mọi dòng


In [16]:
def verdict_of(pred, truth):
    if pred == truth:
        return "OK"
    if pred == predictor.unknown_label:
        return "BỎ SÓT"
    return "SAI"


rows = []
for device, group in sessions[is_field].groupby("canonical_device", sort=True):
    device_rows = [r.to_dict() for _, r in group.iterrows()]
    result = predictor.predict_device(device_rows)
    entry = {"device": device, "n_phien": len(device_rows)}
    for head in predictor.heads:
        truth = group[head].iloc[0]
        pred = result[head]["top1"]
        entry[f"{head}_pred"] = "" if pred == predictor.unknown_label else pred
        entry[f"{head}_truth"] = truth
        entry[f"{head}_verdict"] = verdict_of(pred, truth)
    rows.append(entry)

closedset_report = pd.DataFrame(rows)
closedset_summary = pd.DataFrame({
    head: closedset_report[f"{head}_verdict"].value_counts() for head in predictor.heads
}).T.fillna(0).astype(int)
display(closedset_summary)
display(closedset_report)

,OK
make,12
type,12
model,12


,device,n_phien,make_pred,make_truth,make_verdict,type_pred,type_truth,type_verdict,model_pred,model_truth,model_verdict
0,Desktop PC DNGJHRT,5,Generic Laptop,Generic Laptop,OK,Laptop,Laptop,OK,Windows Desktop HP,Windows Desktop HP,OK
1,Desktop PC DucAnh,6,Generic Laptop,Generic Laptop,OK,Laptop,Laptop,OK,Windows Desktop DELL,Windows Desktop DELL,OK
2,IP Camera (field),6,Camera,Camera,OK,IP Camera,IP Camera,OK,Generic IP Camera,Generic IP Camera,OK
3,Laptop VAF70SQ6,6,Generic Laptop,Generic Laptop,OK,Laptop,Laptop,OK,Windows Laptop HP,Windows Laptop HP,OK
4,Lee Kingdom Laptop,5,Generic Laptop,Generic Laptop,OK,Laptop,Laptop,OK,Windows Desktop DELL,Windows Desktop DELL,OK
5,Linova Laptop (linova),6,Linova/Linux,Linova/Linux,OK,Laptop,Laptop,OK,Linova Laptop HP,Linova Laptop HP,OK
6,OPPO A92,1,OPPO,OPPO,OK,Smartphone,Smartphone,OK,OPPO A92,OPPO A92,OK
7,Raspberry Pi,3,Raspberry Pi,Raspberry Pi,OK,Single-board Computer,Single-board Computer,OK,Raspberry Pi,Raspberry Pi,OK
8,Samsung Phone (Duc Anh),6,Samsung,Samsung,OK,Smartphone,Smartphone,OK,Samsung Galaxy,Samsung Galaxy,OK
9,Xiaomi Redmi Note 10,1,Xiaomi,Xiaomi,OK,Smartphone,Smartphone,OK,Redmi Note 10,Redmi Note 10,OK
